# Encoding Detection & Repair

Standalone module for CSV character-encoding detection (chardet) and mojibake
repair (ftfy), matching `plan.md` Section 4.3 and Section 10 item 6.

This notebook is titled by **content**, not by a task number — elsewhere in
this repo, "Task 5/6" already means rubric dimensions / composite scoring.


## Introduction

Messy client CSVs often arrive without a declared encoding. Decoding as UTF-8
when the file is actually Windows-1252 (or the reverse) produces **mojibake**
(`Ã©` instead of `é`). Phase 1 detects encoding from raw bytes and can repair
already-decoded garbled strings — without AI, fully explainable.


## Methodology

1. **Detect** — `chardet.detect(raw_bytes)` → encoding name + confidence.
2. **Threshold** — if confidence `< 0.8` (`SETTINGS["encoding_confidence_threshold"]`),
   mark `low confidence` and fail the check (plan.md Section 4.3).
3. **Repair** — `ftfy.fix_text` on individual strings; `repair_encoding_frame`
   applies that cell-wise with a change report (never mutates the input frame).

File-level `check_encoding` returns one `CheckResult` with `column=None`.


## Tool Choices & Implementation Notes

**Detection — chardet (≥7)**  
Chosen in plan.md Section 2 item 7: chardet 7 rewrite is ~99% accurate and
much faster than older chardet; flag confidence below 0.8. Runner-up
charset-normalizer was not selected for Phase 1.

**Repair — ftfy**  
Section 2 pairs ftfy specifically for **mojibake repair**, not as a general
encoding converter. Detection names the bytes; ftfy fixes garbled Unicode
after a wrong decode.

**CSV only for check_encoding**  
.xlsx / .xls are already decoded inside openpyxl / xlrd / calamine by the
time a DataFrame exists — there are no meaningful "raw file encoding" bytes
left to sniff on an Excel-sourced frame. Call check_encoding only with CSV
(text) raw bytes during / alongside CSV ingestion. The CLI section is labeled
**Encoding Check (CSV bytes)** (no task number) so it does not collide with
rubric Task 5/6 naming.

**Reconciliation with ingestion.py**  
Previously, engine/ingestion.py had a private _sniff_csv_encoding(path)
that called chardet.detect inline on the first 100,000 bytes. It now calls
check_encoding(sample) and uses 
esult.details["encoding"], preferring

ecommended_encoding when chardet confidence is low and a fallback decode
succeeds. One chardet call path for CSV sniffing + the quality check.

**Sampling**  
check_encoding(..., sample_size=100_000) (settings: encoding_sample_size)
truncates the byte buffer before chardet — same limit historically used by
ingestion — so large CSVs are not fully loaded just to detect encoding.

**Fallback re-decode (low confidence)**  
When confidence < 0.8, try decoding the sample with, in order:
utf-8-sig → cp1252 → latin-1. The first that succeeds is reported as

ecommended_encoding. These three cover BOM-UTF-8 exports, Windows Western
European (common Excel/CSV "ANSI"), and a last-resort single-byte decode.
This does not override a high-confidence chardet result.

**Caching**  
Identical byte samples reuse chardet.detect via unctools.lru_cache on
the sample bytes (_chardet_detect_cached), so ingestion + CLI encoding
check in one process do not double-pay for the same sniff.


## Demo data


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "data_quality_engine").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_quality_engine.config.settings import SETTINGS
from data_quality_engine.engine.checks.encoding import (
    check_encoding,
    repair_encoding,
    repair_encoding_frame,
)

print("confidence threshold =", SETTINGS["encoding_confidence_threshold"])

utf8_bytes = "Order for café — résumé".encode("utf-8")
latin1_bytes = "Café Niño £50".encode("latin-1")
mojibake = "Ã©"  # UTF-8 é mis-decoded
clean_df = pd.DataFrame({"city": ["Lahore", "Karachi"], "qty": [1, 2]})
dirty_df = pd.DataFrame(
    {
        "notes": ["ok", "Ã©lan", "naÃ¯ve"],
        "id": [1, 2, 3],
    }
)


## `check_encoding` — UTF-8 sample


In [ ]:
utf8_result = check_encoding(utf8_bytes)
print(utf8_result)
assert utf8_result.column is None
assert utf8_result.dimension == "consistency"
assert utf8_result.status == "passed"
assert utf8_result.details["low_confidence"] is False
print("encoding:", utf8_result.details["encoding"], "confidence:", utf8_result.details["confidence"])


## `check_encoding` — non-UTF-8 (latin-1) sample


In [ ]:
latin1_result = check_encoding(latin1_bytes)
print(latin1_result)
assert latin1_result.status in {"passed", "failed"}
assert latin1_result.details["low_confidence"] == (latin1_result.status == "failed")
print(
    "encoding:", latin1_result.details.get("encoding"),
    "confidence:", latin1_result.details.get("confidence"),
    "low_confidence:", latin1_result.details.get("low_confidence"),
)


## `repair_encoding` — single string


In [ ]:
fixed = repair_encoding(mojibake)
print(repr(mojibake), "->", repr(fixed))
assert fixed == "é"
assert fixed != mojibake


## `repair_encoding_frame` — transparent batch repair


In [ ]:
out_clean, report_clean = repair_encoding_frame(clean_df)
assert out_clean is not clean_df
assert report_clean["total_cells_changed"] == 0
assert report_clean["columns_changed"] == {}

out_dirty, report_dirty = repair_encoding_frame(dirty_df)
assert out_dirty is not dirty_df
assert report_dirty["total_cells_changed"] >= 1
assert "notes" in report_dirty["columns_changed"]
assert dirty_df.loc[1, "notes"] == "Ã©lan"  # input unchanged
print("report_clean:", report_clean)
print("report_dirty:", report_dirty)
print(out_dirty)


## Validation assertions — bad input never crashes


In [ ]:
err = check_encoding("not-bytes")  # type: ignore[arg-type]
assert err.status == "error"
assert "error" in err.details

empty_df, err_report = repair_encoding_frame(None)  # type: ignore[arg-type]
assert err_report["status"] == "error"
print("soft-fail OK:", err.details["error"], "|", err_report["error"])


## Results table


In [ ]:
rows = [
    {
        "check": "check_encoding (utf-8)",
        "status": utf8_result.status,
        "encoding": utf8_result.details.get("encoding"),
        "confidence": round(float(utf8_result.details.get("confidence") or 0), 4),
        "issues": utf8_result.issues_found,
    },
    {
        "check": "check_encoding (latin-1 bytes)",
        "status": latin1_result.status,
        "encoding": latin1_result.details.get("encoding"),
        "confidence": round(float(latin1_result.details.get("confidence") or 0), 4),
        "issues": latin1_result.issues_found,
    },
    {
        "check": "repair_encoding_frame (dirty)",
        "status": report_dirty.get("status"),
        "encoding": "-",
        "confidence": None,
        "issues": report_dirty.get("total_cells_changed"),
    },
]
pd.DataFrame(rows)


## Limitations

- Excel workbooks are out of scope for file-level encoding detection.
- tfy repairs mojibake heuristics; preferring the correct decode at read
  time (via chardet + fallback recommendation) is still the first line of defense.
- Fallback order is fixed/config-driven; it is not a full encoding classifier.


## Conclusion

engine/checks/encoding.py delivers plan Section 4.3 plus approved extras:
sampled chardet detection, low-confidence fallbacks, process-level cache,
transparent 
epair_encoding_frame, ingestion reconciliation, and a CLI
**Encoding Check (CSV bytes)** section that is not numbered as Task 5/6.
